
# CBIM Problem Canvas
## Mini Use Case 02 — Credit Pre-Screening Engine: Loan Intake Optimisation

**Methodology:** Calibo Business Innovation Methodology (CBIM)  
**Industry:** BFSI — Retail Lending  
**Business Objective:** Optimise loan intake by reducing unnecessary manual reviews while maintaining safe and fair credit decisions.

| CBIM Element | Business Analysis |
|---|---|
| **1. Situation (S)** | A mid-sized Indian private bank receives approximately 3,000 personal loan applications monthly. Every application undergoes manual credit review, requiring approximately 25 minutes per application. |
| **2. Complication (C)** | Nearly half of the applications are declined, and many straightforward declines can be identified quickly through creditworthiness and affordability indicators. However, all applications undergo manual review, increasing analyst workload, operational inefficiency, and potential delays for customers. |
| **3. Question (Q)** | How can the bank use historical credit decisions to safely auto-decline clear-cut applications, fast-track likely approvals, and route uncertain applications to human analysts while reducing manual workload? |
| **4. Single Source of Truth (SSOT)** | The provided simulated dataset contains 12,000 loan applications from April to September 2026. The `decision` column records the historical credit analyst decision (Approved/Declined). It does not represent actual loan repayment success or default risk. |
| **5. Success Definition** | Maximise the percentage of applications removed from full manual review while ensuring that no more than 5% of auto-declined applications were historically approved and no more than 5% of fast-tracked applications were historically declined. Quantify monthly analyst hours saved and assess fairness across applicant groups. |
| **6. Primary Stakeholder** | The Credit Policy Analyst, who recommends the screening policy to the Head of Retail Lending. Other stakeholders include credit analysts, loan applicants, and bank risk and compliance teams. |

### Business Success KPIs

| KPI | Measurement / Target |
|---|---|
| Wrong Auto-Decline Rate | Historically approved among auto-declined applications / Total auto-declined applications × 100 ≤ 5% |
| Wrong Fast-Track Rate | Historically declined among fast-tracked applications / Total fast-tracked applications × 100 ≤ 5% |
| Automation Coverage | (Auto-declined + Fast-tracked applications) / Total applications × 100 |
| Manual Review Reduction | Number of applications removed from full manual review |
| Gross Analyst Hours Saved | Full manual reviews avoided × 25 / 60 |
| Fairness Assessment | Compare historical approval rates and automated decline rates across gender groups |

### Assumptions and Limitations

- Historical analyst decisions are treated as reference labels, but their actual correctness is unknown.
- Actual loan repayment and default outcomes are unavailable.
- The 5% error limits measure disagreement with historical analyst decisions, not actual default risk.
- The fast-track lane still requires a light-touch check, so gross hours saved may exceed actual net time savings.
- Independent validation, fairness checks, and ongoing monitoring are necessary before real-world deployment.



# Part 1 — Baseline Credit Pre-Screener

## 5.1 Load, Inspect and Clean

### Objective

Understand the structure, quality, completeness, and historical approval distribution of the loan application dataset before applying any cleaning transformations.

### Data Quality Checks
- Dataset dimensions and column information
- Data types and missing values
- Duplicate records and application IDs
- Historical loan approval distribution
- Missing income patterns across employment types
- Missing credit scores and credit history
- Unusual or implausible monthly income values

**Data handling principle:** Preserve the raw dataset and document every cleaning decision with a business justification.


In [1]:

from pathlib import Path
import pandas as pd
import numpy as np

# Find project root
PROJECT_ROOT = Path.cwd().resolve()

if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

# Dataset location
DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "MUC02_Credit_PreScreening_Dataset.csv"
)

# Verify dataset exists
assert DATA_PATH.is_file(), f"Dataset not found: {DATA_PATH}"

# Load the dataset
df_raw = pd.read_csv(DATA_PATH)
df = df_raw.copy()

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

display(df.head())


Dataset loaded successfully!
Dataset shape: (12000, 19)


,application_id,application_date,age,gender,city,city_tier,employment_type,years_in_current_job,monthly_income,existing_emi,credit_score,credit_history_months,num_active_loans,delinquencies_24m,loan_purpose,loan_amount,tenure_months,proposed_emi,decision
0,APP100001,2026-04-01,28,Male,Hyderabad,1,Salaried,4.1,95000.0,23500.0,784.0,109,2,0,Home Renovation,240000.0,48,6600.0,Approved
1,APP100002,2026-04-01,36,Female,Raipur,3,Business Owner,0.6,43000.0,8500.0,790.0,162,2,0,Wedding,145000.0,36,5000.0,Approved
2,APP100003,2026-04-01,35,Male,Nagpur,2,Salaried,4.4,35000.0,3500.0,718.0,151,1,1,Medical,345000.0,36,11800.0,Approved
3,APP100004,2026-04-01,42,Male,Jaipur,2,Salaried,5.8,44500.0,0.0,NaN,0,0,0,Travel,135000.0,36,4600.0,Approved
4,APP100005,2026-04-01,34,Male,Kochi,2,Self-Employed Professional,2.1,83500.0,6000.0,700.0,127,1,3,Medical,700000.0,24,33600.0,Declined


In [2]:

print("DATASET INFORMATION")
df.info()

print("\nMISSING VALUES")
display(
    df.isnull().sum()
      .sort_values(ascending=False)
      .to_frame("missing_count")
)

print("\nDUPLICATE RECORDS")
print("Full-row duplicates:", df.duplicated().sum())

print(
    "Duplicate application IDs:",
    df["application_id"].duplicated().sum()
)

print("\nTARGET DISTRIBUTION")
display(
    df["decision"]
      .value_counts(dropna=False)
      .to_frame("count")
)

print("\nAPPROVAL PERCENTAGE")
print(
    df["decision"]
      .eq("Approved")
      .mean() * 100
)


DATASET INFORMATION
<class 'pandas.DataFrame'>
RangeIndex: 12000 entries, 0 to 11999
Data columns (total 19 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   application_id         12000 non-null  str    
 1   application_date       12000 non-null  str    
 2   age                    12000 non-null  int64  
 3   gender                 12000 non-null  str    
 4   city                   12000 non-null  str    
 5   city_tier              12000 non-null  int64  
 6   employment_type        12000 non-null  str    
 7   years_in_current_job   12000 non-null  float64
 8   monthly_income         11559 non-null  float64
 9   existing_emi           12000 non-null  float64
 10  credit_score           10961 non-null  float64
 11  credit_history_months  12000 non-null  int64  
 12  num_active_loans       12000 non-null  int64  
 13  delinquencies_24m      12000 non-null  int64  
 14  loan_purpose           12000 non-null  str   

,missing_count
credit_score,1039
monthly_income,441
application_id,0
age,0
application_date,0
gender,0
city,0
employment_type,0
city_tier,0
years_in_current_job,0



DUPLICATE RECORDS
Full-row duplicates: 0
Duplicate application IDs: 0

TARGET DISTRIBUTION


,count
decision,
Approved,6353
Declined,5647



APPROVAL PERCENTAGE
52.94166666666666


In [3]:
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
application_id,12000,12000,APP100001,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
application_date,12000,183,2026-06-28,98,NaN,NaN,NaN,NaN,NaN,NaN,NaN
age,12000.0,NaN,NaN,NaN,33.15375,7.058878,21.0,28.0,33.0,38.0,60.0
gender,12000,2,Male,8205,NaN,NaN,NaN,NaN,NaN,NaN,NaN
city,12000,26,Bengaluru,746,NaN,NaN,NaN,NaN,NaN,NaN,NaN
city_tier,12000.0,NaN,NaN,NaN,1.707917,0.757728,1.0,1.0,2.0,2.0,3.0
employment_type,12000,4,Salaried,7057,NaN,NaN,NaN,NaN,NaN,NaN,NaN
years_in_current_job,12000.0,NaN,NaN,NaN,3.77485,3.149578,0.0,1.5,3.0,5.1,30.0
monthly_income,11559.0,NaN,NaN,NaN,60314.862877,48890.827938,12000.0,33500.0,50500.0,74750.0,1656000.0
existing_emi,12000.0,NaN,NaN,NaN,6273.166667,8526.1693,0.0,0.0,4000.0,8500.0,96500.0



### Initial Data Quality Findings

- **Total applications:** 12,000
- **Approved applications:** 6,353 (52.94%)
- **Declined applications:** 5,647 (47.06%)
- **Full-row duplicates:** 0
- **Duplicate application IDs:** 0

**Business Interpretation:**

The historical approval and decline classes are reasonably balanced, with a slight majority of approved applications. Severe class imbalance is not evident, so class resampling is not required at this stage.

No duplicate records or application identifiers were detected. The target distribution provides a useful starting point for supervised classification.

However, model accuracy alone will not determine business success. The final credit pre-screening solution must satisfy the bank's two 5% automated-decision error limits.


In [4]:

# Missing monthly income by employment type

income_missing = (
    df.groupby("employment_type")
      .agg(
          total_applications=("monthly_income", "size"),
          missing_income=("monthly_income", lambda x: x.isna().sum()),
          median_income=("monthly_income", "median")
      )
)

income_missing["missing_percentage"] = (
    income_missing["missing_income"]
    / income_missing["total_applications"] * 100
)

display(
    income_missing.sort_values(
        "missing_percentage",
        ascending=False
    ).round(2)
)

print(
    "Total missing income:",
    df["monthly_income"].isna().sum()
)

print(
    "\nMaximum reported monthly income:",
    df["monthly_income"].max()
)


,total_applications,missing_income,median_income,missing_percentage
employment_type,,,,
Business Owner,1749,227,69500.0,12.98
Self-Employed Professional,1428,154,75000.0,10.78
Salaried,7057,49,50000.0,0.69
Gig / Contract,1766,11,28000.0,0.62


Total missing income: 441

Maximum reported monthly income: 1656000.0


In [5]:

# Investigate missing credit scores

missing_score = df[df["credit_score"].isna()]

print("Missing credit scores:", len(missing_score))

print(
    "Missing percentage:",
    round(df["credit_score"].isna().mean() * 100, 2)
)

print("\nCredit history for missing-score applicants:")
display(
    missing_score["credit_history_months"]
    .value_counts(dropna=False)
    .sort_index()
)

print(
    "\nMissing scores with zero credit history:",
    (missing_score["credit_history_months"] == 0).sum()
)

print(
    "\nMissing scores with non-zero credit history:",
    (missing_score["credit_history_months"] > 0).sum()
)


Missing credit scores: 1039
Missing percentage: 8.66

Credit history for missing-score applicants:


credit_history_months
0    1039
Name: count, dtype: int64


Missing scores with zero credit history: 1039

Missing scores with non-zero credit history: 0


In [6]:


print("\nIncome distribution:")
display(
    df.groupby("employment_type")["monthly_income"]
      .describe(percentiles=[0.5, 0.9, 0.95, 0.99])
      .round(2)
)



Income distribution:


,count,mean,std,min,50%,90%,95%,99%,max
employment_type,,,,,,,,,
Business Owner,1522.0,79441.52,43398.78,12000.0,69500.0,136500.0,162475.0,227395.0,330000.0
Gig / Contract,1755.0,32152.14,17619.94,12000.0,28000.0,53500.0,66500.0,97230.0,170000.0
Salaried,7008.0,58827.34,52174.95,12000.0,50000.0,95650.0,115825.0,170430.0,1656000.0
Self-Employed Professional,1274.0,84443.09,43756.96,12000.0,75000.0,140500.0,164850.0,220390.0,349000.0


In [7]:
income = df["monthly_income"].dropna()

q1 = income.quantile(0.25)
q3 = income.quantile(0.75)

iqr = q3 - q1
upper_fence = q3 + 1.5 * iqr

print("Q1:", q1)
print("Q3:", q3)
print("IQR:", iqr)
print("Upper fence:", upper_fence)

print(
    "Potential outliers:",
    (income > upper_fence).sum()
)

Q1: 33500.0
Q3: 74750.0
IQR: 41250.0
Upper fence: 136625.0
Potential outliers: 473


In [8]:

# Examine income percentiles
print("INCOME PERCENTILES")

display(
    df["monthly_income"]
    .quantile([0.50, 0.75, 0.90, 0.95, 0.99, 0.995, 0.999, 1.0])
    .to_frame("monthly_income")
)

# Inspect the 20 highest incomes
print("\nTOP 20 HIGHEST INCOMES")

display(
    df[
        [
            "application_id",
            "employment_type",
            "monthly_income",
            "existing_emi",
            "loan_amount",
            "decision"
        ]
    ]
    .sort_values("monthly_income", ascending=False)
    .head(20)
)


INCOME PERCENTILES


,monthly_income
0.500,50500.0
0.750,74750.0
0.900,106000.0
0.950,130000.0
0.990,191000.0
0.995,234210.0
0.999,577824.0
1.000,1656000.0



TOP 20 HIGHEST INCOMES


,application_id,employment_type,monthly_income,existing_emi,loan_amount,decision
1467,APP101468,Salaried,1656000.0,16500.0,925000.0,Declined
300,APP100301,Salaried,1254000.0,0.0,240000.0,Approved
2493,APP102494,Salaried,1212000.0,23500.0,600000.0,Approved
589,APP100590,Salaried,1068000.0,13500.0,600000.0,Declined
9849,APP109850,Salaried,990000.0,21000.0,260000.0,Declined
1848,APP101849,Salaried,984000.0,0.0,500000.0,Approved
2488,APP102489,Salaried,972000.0,0.0,310000.0,Approved
4010,APP104011,Salaried,900000.0,4000.0,580000.0,Approved
4898,APP104899,Salaried,870000.0,4500.0,645000.0,Declined
9261,APP109262,Salaried,702000.0,0.0,495000.0,Approved


In [9]:

# Overall IQR threshold applied to each employment group
upper_fence = 136625.0

df["income_outlier"] = df["monthly_income"] > upper_fence

outlier_summary = (
    df.groupby("employment_type")
    .agg(
        total_applications=("monthly_income", "size"),
        valid_incomes=("monthly_income", "count"),
        outlier_count=("income_outlier", "sum"),
        maximum_income=("monthly_income", "max"),
        median_income=("monthly_income", "median")
    )
)

outlier_summary["outlier_percentage"] = (
    outlier_summary["outlier_count"]
    / outlier_summary["valid_incomes"] * 100
)

display(
    outlier_summary.sort_values(
        "outlier_percentage",
        ascending=False
    ).round(2)
)


,total_applications,valid_incomes,outlier_count,maximum_income,median_income,outlier_percentage
employment_type,,,,,,
Self-Employed Professional,1428,1274,149,349000.0,75000.0,11.70
Business Owner,1749,1522,150,330000.0,69500.0,9.86
Salaried,7057,7008,172,1656000.0,50000.0,2.45
Gig / Contract,1766,1755,2,170000.0,28000.0,0.11


In [10]:

# Examine income distribution for salaried applicants

salaried = df.loc[
    df["employment_type"] == "Salaried",
    "monthly_income"
].dropna()

print("SALARIED INCOME PERCENTILES")

display(
    salaried.quantile(
        [0.50, 0.75, 0.90, 0.95, 0.99, 0.995, 0.999, 1.0]
    ).to_frame("monthly_income")
)

print("\nSALARIED INCOME BANDS")

bands = [0, 100000, 200000, 300000, 400000,
         500000, 750000, 1000000, float("inf")]

labels = [
    "0–1L", "1–2L", "2–3L", "3–4L",
    "4–5L", "5–7.5L", "7.5–10L", "10L+"
]

income_bands = pd.cut(
    salaried,
    bins=bands,
    labels=labels,
    include_lowest=True
)

display(
    income_bands.value_counts()
                .sort_index()
                .to_frame("applicants")
)


SALARIED INCOME PERCENTILES


,monthly_income
0.500,50000.0
0.750,71000.0
0.900,95650.0
0.950,115825.0
0.990,170430.0
0.995,230115.0
0.999,899790.0
1.000,1656000.0



SALARIED INCOME BANDS


,applicants
monthly_income,
0–1L,6415
1–2L,552
2–3L,19
3–4L,5
4–5L,4
5–7.5L,4
7.5–10L,5
10L+,4


In [11]:

#
extreme_salaried = df.loc[
    (df["employment_type"] == "Salaried")
    & (df["monthly_income"] > 500000)
].copy()

extreme_salaried["possible_monthly_if_annual"] = (
    extreme_salaried["monthly_income"] / 12
)

extreme_salaried["foir_as_recorded"] = (
    (extreme_salaried["existing_emi"]
     + extreme_salaried["proposed_emi"])
    / extreme_salaried["monthly_income"]
)

extreme_salaried["foir_if_annual"] = (
    (extreme_salaried["existing_emi"]
     + extreme_salaried["proposed_emi"])
    / extreme_salaried["possible_monthly_if_annual"]
)

display(
    extreme_salaried[
        [
            "application_id",
            "monthly_income",
            "possible_monthly_if_annual",
            "foir_as_recorded",
            "foir_if_annual",
            "decision"
        ]
    ].sort_values(
        "monthly_income", ascending=False
    ).round(3)
)


,application_id,monthly_income,possible_monthly_if_annual,foir_as_recorded,foir_if_annual,decision
1467,APP101468,1656000.0,138000.0,0.037,0.441,Declined
300,APP100301,1254000.0,104500.0,0.005,0.063,Approved
2493,APP102494,1212000.0,101000.0,0.031,0.371,Approved
589,APP100590,1068000.0,89000.0,0.026,0.309,Declined
9849,APP109850,990000.0,82500.0,0.034,0.406,Declined
1848,APP101849,984000.0,82000.0,0.017,0.209,Approved
2488,APP102489,972000.0,81000.0,0.011,0.131,Approved
4010,APP104011,900000.0,75000.0,0.026,0.317,Approved
4898,APP104899,870000.0,72500.0,0.022,0.269,Declined
9261,APP109262,702000.0,58500.0,0.024,0.289,Approved


In [12]:

# Examine other credit-risk indicators for the 13 applicants

columns_to_check = [
    "application_id",
    "monthly_income",
    "credit_score",
    "credit_history_months",
    "delinquencies_24m",
    "num_active_loans",
    "years_in_current_job",
    "decision"
]

display(
    extreme_salaried[columns_to_check]
    .sort_values("monthly_income", ascending=False)
)


,application_id,monthly_income,credit_score,credit_history_months,delinquencies_24m,num_active_loans,years_in_current_job,decision
1467,APP101468,1656000.0,718.0,118,0,1,3.9,Declined
300,APP100301,1254000.0,NaN,0,0,0,7.6,Approved
2493,APP102494,1212000.0,718.0,240,0,2,8.8,Approved
589,APP100590,1068000.0,700.0,106,3,2,1.4,Declined
9849,APP109850,990000.0,633.0,104,3,2,1.0,Declined
1848,APP101849,984000.0,NaN,0,0,0,2.5,Approved
2488,APP102489,972000.0,695.0,204,0,0,9.4,Approved
4010,APP104011,900000.0,789.0,287,0,1,10.7,Approved
4898,APP104899,870000.0,682.0,189,2,1,4.1,Declined
9261,APP109262,702000.0,740.0,87,1,0,6.6,Approved


In [13]:
applicant = df.loc[
    df["application_id"] == "APP101468"
]

display(applicant.T)

,1467
application_id,APP101468
application_date,2026-04-23
age,37
gender,Male
city,Lucknow
city_tier,2
employment_type,Salaried
years_in_current_job,3.9
monthly_income,1656000.0
existing_emi,16500.0


In [14]:

# Employment-specific income outlier analysis

def income_outlier_summary(group):
    income = group["monthly_income"].dropna()

    q1 = income.quantile(0.25)
    q3 = income.quantile(0.75)
    iqr = q3 - q1
    upper_fence = q3 + 1.5 * iqr

    return pd.Series({
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "Upper_Fence": upper_fence,
        "Outlier_Count": (income > upper_fence).sum(),
        "Outlier_Rate_%": (
            (income > upper_fence).mean() * 100
        ),
        "Maximum_Income": income.max()
    })

group_outliers = (
    df.groupby("employment_type")
      .apply(income_outlier_summary)
)

display(group_outliers.round(2))


,Q1,Q3,IQR,Upper_Fence,Outlier_Count,Outlier_Rate_%,Maximum_Income
employment_type,,,,,,,
Business Owner,49000.0,98500.0,49500.0,172750.0,56.0,3.68,330000.0
Gig / Contract,20000.0,39000.0,19000.0,67500.0,83.0,4.73,170000.0
Salaried,35500.0,71000.0,35500.0,124250.0,268.0,3.82,1656000.0
Self-Employed Professional,53000.0,106875.0,53875.0,187687.5,35.0,2.75,349000.0


In [15]:

# Inspect top 10 incomes for each non-salaried group

for employment in [
    "Self-Employed Professional",
    "Business Owner",
    "Gig / Contract"
]:
    print(f"\n{employment.upper()} — TOP 10 INCOMES")

    display(
        df.loc[
            df["employment_type"] == employment,
            [
                "application_id",
                "monthly_income",
                "existing_emi",
                "credit_score",
                "loan_amount",
                "decision"
            ]
        ]
        .sort_values("monthly_income", ascending=False)
        .head(10)
    )



SELF-EMPLOYED PROFESSIONAL — TOP 10 INCOMES


,application_id,monthly_income,existing_emi,credit_score,loan_amount,decision
7276,APP107277,349000.0,38000.0,701.0,1405000.0,Approved
121,APP100122,338000.0,80000.0,794.0,1580000.0,Approved
9461,APP109462,324000.0,39500.0,727.0,2490000.0,Approved
9275,APP109276,297500.0,18000.0,713.0,2240000.0,Approved
5414,APP105415,271500.0,29500.0,703.0,2350000.0,Approved
11432,APP111433,265000.0,0.0,619.0,1525000.0,Declined
7370,APP107371,255500.0,32000.0,724.0,2315000.0,Approved
4546,APP104547,255500.0,0.0,NaN,770000.0,Declined
2895,APP102896,255500.0,0.0,753.0,1715000.0,Declined
6251,APP106252,254500.0,0.0,NaN,1130000.0,Declined



BUSINESS OWNER — TOP 10 INCOMES


,application_id,monthly_income,existing_emi,credit_score,loan_amount,decision
2499,APP102500,330000.0,34000.0,711.0,2500000.0,Approved
2698,APP102699,320000.0,43500.0,783.0,1630000.0,Approved
6166,APP106167,312000.0,22500.0,755.0,2500000.0,Approved
4654,APP104655,282500.0,24500.0,689.0,645000.0,Declined
9527,APP109528,278000.0,22000.0,666.0,2500000.0,Approved
2883,APP102884,272500.0,0.0,NaN,1170000.0,Approved
11343,APP111344,268500.0,62500.0,824.0,2500000.0,Approved
6027,APP106028,264000.0,58500.0,676.0,1900000.0,Approved
6490,APP106491,263500.0,46500.0,734.0,725000.0,Declined
5335,APP105336,246500.0,17000.0,635.0,2500000.0,Declined



GIG / CONTRACT — TOP 10 INCOMES


,application_id,monthly_income,existing_emi,credit_score,loan_amount,decision
7558,APP107559,170000.0,29000.0,727.0,575000.0,Declined
4133,APP104134,150000.0,26000.0,591.0,1260000.0,Declined
5138,APP105139,133500.0,13500.0,751.0,390000.0,Approved
9018,APP109019,129000.0,0.0,755.0,1230000.0,Approved
4053,APP104054,118000.0,10000.0,679.0,725000.0,Declined
4967,APP104968,117500.0,8500.0,624.0,780000.0,Declined
5398,APP105399,117500.0,0.0,598.0,675000.0,Declined
5912,APP105913,115000.0,21500.0,694.0,495000.0,Declined
10064,APP110065,113000.0,43500.0,764.0,1085000.0,Declined
9017,APP109018,112000.0,10500.0,878.0,1060000.0,Declined



### 5.1.1 Missing Value Analysis — Findings and Cleaning Decisions

The dataset contains 12,000 historical loan applications. Missing values were investigated to understand whether they represent data-quality problems or meaningful business information.

#### A. Missing Monthly Income

**Findings**

A total of **441 applications (3.68%)** have missing monthly income values.

| Employment Type | Total Applications | Missing Income | Missing Rate | Recorded Median Income |
|---|---:|---:|---:|---:|
| Business Owner | 1,749 | 227 | 12.98% | ₹69,500 |
| Self-Employed Professional | 1,428 | 154 | 10.78% | ₹75,000 |
| Salaried | 7,057 | 49 | 0.69% | ₹50,000 |
| Gig / Contract | 1,766 | 11 | 0.62% | ₹28,000 |
| **Total** | **12,000** | **441** | **3.68%** | — |

**Business Interpretation**

Missing income values are disproportionately concentrated among Business Owners and Self-Employed Professionals.

Together, these two categories account for 381 of the 441 missing income records (86.39%).

This indicates that income completeness differs considerably across employment categories. However, the actual reason for the missing values cannot be established from the available data.

The employment groups also have different income distributions, making a single overall income-imputation value less representative.

**Proposed Cleaning Decision**

1. Retain all applications with missing monthly income.
2. Use employment-specific median imputation as the primary candidate treatment.
3. Calculate median values using training data only to avoid data leakage.
4. Include a fallback training-set median for previously unseen employment categories.
5. Preserve an indicator of originally missing income where useful for data-quality monitoring.

**Business Justification**

Median imputation is more robust to unusually high incomes than mean imputation. Employment-specific medians better reflect differences between salaried employees, business owners, self-employed professionals, and gig workers.

However, imputed income is an estimate and must not be treated as verified income in real lending decisions.

---

#### B. Missing Credit Scores

**Findings**

A total of **1,039 applications (8.66%)** have missing credit scores.

| Investigation | Result |
|---|---:|
| Total applications | 12,000 |
| Missing credit scores | 1,039 |
| Missing percentage | 8.66% |
| Missing-score applicants with zero credit-history months | 1,039 |
| Missing-score applicants with positive credit-history months | 0 |

All applicants with missing credit scores have zero recorded months of credit history.

**Business Interpretation**

The pattern strongly supports the interpretation that these applicants are **New-to-Credit (NTC)** rather than records with accidental missing credit scores.

This distinction is important because the absence of a credit score does not automatically indicate poor creditworthiness.

Dropping these applications would exclude a meaningful customer segment and could introduce selection bias.

**Proposed Cleaning Decision**

1. Retain all 1,039 applicants with missing credit scores.
2. Create `is_ntc = 1` when the original credit score is missing, otherwise `0`.
3. Create the NTC indicator before imputing the credit score.
4. Impute missing credit scores using the training-set median within the modelling pipeline.
5. Preserve the NTC indicator so the model can distinguish an imputed score from an originally observed score.

**Business Justification**

The credit-score median is used only to provide a valid numerical input for the model. It is not intended to represent the applicant's actual bureau score.

The `is_ntc` indicator preserves the business meaning of missing credit history and allows the model to learn differences between new-to-credit and established-credit applicants.

New-to-credit customers should not be automatically declined solely because they lack a bureau score.

---

#### C. Overall Missing-Value Treatment

| Feature | Missing Records | Proposed Treatment | Business Reason |
|---|---:|---|---|
| `monthly_income` | 441 | Employment-specific training median | Preserve applications while accounting for employment-related income differences |
| `credit_score` | 1,039 | Training median + `is_ntc` flag | Preserve new-to-credit applicants and the meaning of missing scores |

**Data Leakage Prevention**

All data-derived imputation statistics will be fitted on the training partition only. During cross-validation, these statistics will be learned separately within each training fold through the preprocessing pipeline.




In [16]:

# Preserve raw observations
df_clean = df_raw.copy()

# Creating indicators before any imputation
df_clean["income_was_missing"] = (
    df_clean["monthly_income"].isna().astype(int)
)

df_clean["is_ntc"] = (
    df_clean["credit_score"].isna().astype(int)
)

print("Missing income flags:",
      df_clean["income_was_missing"].sum())

print("New-to-credit flags:",
      df_clean["is_ntc"].sum())

print("Missing income values:",
      df_clean["monthly_income"].isna().sum())

print("Missing credit scores:",
      df_clean["credit_score"].isna().sum())


Missing income flags: 441
New-to-credit flags: 1039
Missing income values: 441
Missing credit scores: 1039


In [17]:

credit_history_check = pd.crosstab(
    df_clean["credit_history_months"].eq(0),
    df_clean["credit_score"].isna(),
    rownames=["Zero credit history"],
    colnames=["Credit score missing"]
)

display(credit_history_check)


Credit score missing,False,True
Zero credit history,,
False,10961,0
True,0,1039



### 5.1.2 Cleaning Preparation and Missingness Indicators

The original dataset has been preserved in `df_raw`, while a separate working DataFrame, `df_clean`, has been created for preprocessing.

Two missingness indicators have been introduced:

- `income_was_missing`: Identifies applicants whose monthly income was originally missing.
- `is_ntc`: Identifies applicants whose credit score was missing.

The indicators preserve information that would otherwise be lost during imputation.

**Missing-value strategy:**

Monthly income will be imputed using employment-specific medians, while missing credit scores will be imputed using the training-set median.

These statistics will be learned only from the training partition and separately within cross-validation folds.

**Outlier strategy:**

Observed income values remain unchanged. Extreme salaried incomes suspected of annual-versus-monthly reporting inconsistencies are retained and documented for verification.

**Current status:** Cleaning indicators have been prepared. Numerical imputation remains deferred to the modelling pipeline to prevent data leakage.


**Additional Data Consistency Validation**

A cross-tabulation between zero credit-history months and missing credit scores confirmed a perfect correspondence:

- 1,039 applicants have zero credit-history months and missing credit scores.
- 10,961 applicants have positive credit history and recorded credit scores.
- No inconsistent combinations were observed.

This confirms that the missing credit-score pattern aligns completely with the New-to-Credit (NTC) definition in the provided dataset.

**Cleaning Decision:** Retain all NTC applicants and preserve their status through the `is_ntc` indicator. Missing credit scores will be imputed within the training pipeline rather than modified in the raw dataset.


In [18]:

# Step 5.1 — Data type validation

print("FEATURE DATA TYPES")
display(
    df_clean.dtypes.to_frame("current_dtype")
)

print("\nNUMERIC COLUMN SUMMARY")
display(
    df_clean.select_dtypes(include="number")
    .describe()
    .T
)


FEATURE DATA TYPES


,current_dtype
application_id,str
application_date,str
age,int64
gender,str
city,str
city_tier,int64
employment_type,str
years_in_current_job,float64
monthly_income,float64
existing_emi,float64



NUMERIC COLUMN SUMMARY


,count,mean,std,min,25%,50%,75%,max
age,12000.0,33.153750,7.058878,21.0,28.0,33.0,38.0,60.0
city_tier,12000.0,1.707917,0.757728,1.0,1.0,2.0,2.0,3.0
years_in_current_job,12000.0,3.774850,3.149578,0.0,1.5,3.0,5.1,30.0
monthly_income,11559.0,60314.862877,48890.827938,12000.0,33500.0,50500.0,74750.0,1656000.0
existing_emi,12000.0,6273.166667,8526.169300,0.0,0.0,4000.0,8500.0,96500.0
credit_score,10961.0,725.920080,62.559023,479.0,684.0,726.0,769.0,890.0
credit_history_months,12000.0,114.880167,77.329483,0.0,54.0,114.0,169.0,395.0
num_active_loans,12000.0,1.166000,1.148356,0.0,0.0,1.0,2.0,8.0
delinquencies_24m,12000.0,0.775750,1.227024,0.0,0.0,0.0,1.0,11.0
loan_amount,12000.0,379592.500000,295073.073639,50000.0,175000.0,300000.0,490000.0,2500000.0


In [19]:

# Business-rule validation — investigation only

checks = {
    "Invalid age (<18 or >100)": (
        (df_clean["age"] < 18) |
        (df_clean["age"] > 100)
    ),
    "Non-positive recorded income": (
        df_clean["monthly_income"].notna() &
        (df_clean["monthly_income"] <= 0)
    ),
    "Negative existing EMI": (
        df_clean["existing_emi"] < 0
    ),
    "Negative proposed EMI": (
        df_clean["proposed_emi"] < 0
    ),
    "Non-positive loan amount": (
        df_clean["loan_amount"] <= 0
    ),
    "Non-positive loan tenure": (
        df_clean["tenure_months"] <= 0
    ),
    "Credit score outside 300–900": (
        df_clean["credit_score"].notna() &
        ~df_clean["credit_score"].between(300, 900)
    ),
    "Negative credit history": (
        df_clean["credit_history_months"] < 0
    ),
    "Negative active loans": (
        df_clean["num_active_loans"] < 0
    ),
    "Negative delinquencies": (
        df_clean["delinquencies_24m"] < 0
    ),
    "Negative job tenure": (
        df_clean["years_in_current_job"] < 0
    )
}

validation_results = pd.DataFrame({
    "validation_rule": list(checks.keys()),
    "invalid_records": [
        int(mask.sum()) for mask in checks.values()
    ]
})

display(validation_results)


,validation_rule,invalid_records
0,Invalid age (<18 or >100),0
1,Non-positive recorded income,0
2,Negative existing EMI,0
3,Negative proposed EMI,0
4,Non-positive loan amount,0
5,Non-positive loan tenure,0
6,Credit score outside 300–900,0
7,Negative credit history,0
8,Negative active loans,0
9,Negative delinquencies,0


In [20]:

# Validate application dates

parsed_dates = pd.to_datetime(
    df_clean["application_date"],
    format="%Y-%m-%d",
    errors="coerce"
)

print("Invalid dates:", parsed_dates.isna().sum())
print("Earliest date:", parsed_dates.min())
print("Latest date:", parsed_dates.max())

print(
    "Dates outside Apr-Sep 2026:",
    (~parsed_dates.between(
        "2026-04-01", "2026-09-30"
    )).sum()
)

# Check categorical values
for col in [
    "gender",
    "city_tier",
    "employment_type",
    "loan_purpose",
    "decision"
]:
    print(f"\n{col}")
    display(df_clean[col].value_counts(dropna=False))


Invalid dates: 0
Earliest date: 2026-04-01 00:00:00
Latest date: 2026-09-30 00:00:00
Dates outside Apr-Sep 2026: 0

gender


gender
Male      8205
Female    3795
Name: count, dtype: int64


city_tier


city_tier
1    5709
2    4087
3    2204
Name: count, dtype: int64


employment_type


employment_type
Salaried                      7057
Gig / Contract                1766
Business Owner                1749
Self-Employed Professional    1428
Name: count, dtype: int64


loan_purpose


loan_purpose
Debt Consolidation    2370
Home Renovation       2157
Wedding               1830
Education             1570
Consumer Durables     1526
Medical               1459
Travel                1088
Name: count, dtype: int64


decision


decision
Approved    6353
Declined    5647
Name: count, dtype: int64

In [21]:
display(df_clean["loan_purpose"].value_counts(dropna=False))
display(df_clean["decision"].value_counts(dropna=False))

loan_purpose
Debt Consolidation    2370
Home Renovation       2157
Wedding               1830
Education             1570
Consumer Durables     1526
Medical               1459
Travel                1088
Name: count, dtype: int64

decision
Approved    6353
Declined    5647
Name: count, dtype: int64


### 5.1 — Data Validation Summary

#### 1. Data Types and Structural Validation

The dataset contains 12,000 loan applications with numerical, categorical, date, and identifier features.

- Numerical features were loaded with appropriate integer or floating-point types.
- Application identifiers are unique.
- No duplicate records or duplicate application IDs were detected.
- The `application_date` field was initially stored as a string and successfully validated using datetime parsing.

#### 2. Numerical Business-Rule Validation

Eleven validation rules were applied to check the validity of numerical features.

| Validation Category | Validation Rule | Invalid Records |
|---|---|---:|
| Applicant Age | Between 18 and 100 years | 0 |
| Monthly Income | Positive where recorded | 0 |
| Existing and Proposed EMI | Non-negative | 0 |
| Loan Amount | Positive | 0 |
| Loan Tenure | Positive | 0 |
| Credit Score | Between 300 and 900 where recorded | 0 |
| Credit History | Non-negative | 0 |
| Active Loans | Non-negative | 0 |
| Delinquencies | Non-negative | 0 |
| Employment Tenure | Non-negative | 0 |

**Finding:** All 11 numerical validation checks passed. No invalid records were identified under the defined rules.

#### 3. Date Validation

| Check | Result |
|---|---|
| Invalid dates | 0 |
| Earliest application | 2026-04-01 |
| Latest application | 2026-09-30 |
| Dates outside the expected period | 0 |

**Finding:** All application dates fall within the expected six-month period (April–September 2026).

#### 4. Categorical Validation

| Feature | Validated Categories | Finding |
|---|---|---|
| Gender | Male, Female | No missing or unexpected categories |
| City Tier | 1, 2, 3 | No invalid tiers |
| Employment Type | Salaried, Business Owner, Self-Employed Professional, Gig / Contract | Four consistent categories |
| Loan Purpose | Debt Consolidation, Home Renovation, Wedding, Education, Consumer Durables, Medical, Travel | Seven consistent categories |
| Decision | Approved, Declined | Two valid target classes |

**Target Distribution:**
- Approved: 6,353 (52.94%)
- Declined: 5,647 (47.06%)

The target is reasonably balanced, with a slight majority of approved applications.

#### 5. Business Interpretation

The dataset passed the numerical, categorical, and date validation checks performed. No invalid values were identified under these rules.

However, passing validation does not guarantee that all recorded information is accurate.

The dataset still contains missing income values, missing credit scores associated with new-to-credit applicants, and unusually high incomes that may require verification.

The dataset also contains more male than female applicants, which will be considered during the fairness analysis.

#### 6. Cleaning Decisions

- Retain all records that passed the validation rules.
- Preserve original categorical values without modification.
- Convert `application_date` to datetime for analysis.
- Do not remove records solely because they are statistical outliers.
- Handle missing income and credit scores using the separately documented preprocessing strategy.
- Fit imputation statistics using training data only to prevent leakage.

**Conclusion:** Structural and business-rule validation is complete. Missing-value imputation and final cleaning verification remain pending.


In [22]:

import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_is_fitted


class EmploymentIncomeImputer(BaseEstimator, TransformerMixin):
    """Fill missing income using employment-specific training medians."""

    def fit(self, X, y=None):
        X = X.copy()

        self.group_medians_ = (
            X.groupby("employment_type")["monthly_income"]
            .median()
        )

        self.global_median_ = X["monthly_income"].median()

        if pd.isna(self.global_median_):
            raise ValueError("Cannot fit: all training incomes are missing.")

        return self

    def transform(self, X):
        check_is_fitted(self, ["group_medians_", "global_median_"])

        X = X.copy()

        group_medians = X["employment_type"].map(
            self.group_medians_
        )

        X["monthly_income"] = (
            X["monthly_income"]
            .fillna(group_medians)
            .fillna(self.global_median_)
        )

        return X


In [23]:
import sys
from pathlib import Path

print("Python executable:", sys.executable)
print("Working directory:", Path.cwd())

Python executable: c:\Users\ASUS\Downloads\prescreening_MLProject\credit-prescreening-ml\.venv\Scripts\python.exe
Working directory: c:\Users\ASUS\Downloads\prescreening_MLProject\credit-prescreening-ml\notebooks


In [24]:

from src.data.imputation import EmploymentIncomeImputer


# Example training records
sample_train = pd.DataFrame({
    "employment_type": [
        "Salaried", "Salaried", "Salaried",
        "Business Owner", "Business Owner", "Business Owner"
    ],
    "monthly_income": [
        40000, 50000, 60000,
        70000, 90000, 110000
    ]
})

# Example unseen records
sample_test = pd.DataFrame({
    "employment_type": [
        "Salaried", "Business Owner", "Gig / Contract"
    ],
    "monthly_income": [None, None, None]
})

imputer = EmploymentIncomeImputer()

# Learn from training records only
imputer.fit(sample_train)

# Apply learned medians to unseen records
result = imputer.transform(sample_test)

print("Learned employment medians:")
display(imputer.group_medians_)

print("\nImputed unseen records:")
display(result)


Learned employment medians:


employment_type
Business Owner    90000.0
Salaried          50000.0
Name: monthly_income, dtype: float64


Imputed unseen records:


,employment_type,monthly_income
0,Salaried,50000.0
1,Business Owner,90000.0
2,Gig / Contract,65000.0



### 5.1.5 — Cleaning Implementation and Testing

A reusable `EmploymentIncomeImputer` was implemented in `src/data/imputation.py`.

The transformer learns employment-specific median incomes from training data and applies those statistics to missing income values. An overall training median provides fallback handling for unseen employment categories.

**Validation:** Automated pytest tests passed, confirming that:
- Missing incomes receive the appropriate group median.
- Previously observed income values remain unchanged.
- Unseen data does not update learned imputation statistics.

The original dataset remains unchanged. Actual imputation will be integrated into the modelling pipeline to prevent data leakage.

Missing credit scores will be handled using training-set median imputation while preserving the `is_ntc` indicator.

**Limitation:** Unverified extreme incomes remain unchanged, and imputed income cannot substitute for verified income during real lending decisions.
